<a href="https://colab.research.google.com/github/aniray2908/satellite-esg-risk-engine/blob/main/experiments/python/ceri/ceri_v3_weight_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CERI v3 — Data-Driven Weight Optimization

This notebook optimizes composite weights by maximizing
cluster separation measured via silhouette score.

Objective:
Select weight vector (w1, w2, w3) that produces
maximum geometric separation in standardized feature space.

k = 3 is used based on prior validation.

In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [3]:
feature_df = pd.read_csv(
    "/content/drive/MyDrive/ceri_v2_feature_layer.csv"
)

feature_df.head()

,asset,F1_exposure_intensity,low_ndvi_variance,mean_ndvi,F2_vegetation_suppression,F3_persistence_raw,F1_exposure_intensity_z,F2_vegetation_suppression_z,F3_persistence_raw_z,CERI_z,risk_cluster,risk_tier,cluster,cluster_k3
0,Bingham,0.999922,2.805335e-09,0.021179,0.978821,1.000000e+00,0.629904,0.421693,0.519958,0.545452,0,High Risk,0,2
1,Carajás,0.937332,2.050342e-03,0.035150,0.964850,4.877238e-07,0.213303,0.268879,-1.499415,-0.112568,0,High Risk,0,0
2,Gevra,0.684294,6.057728e-05,0.193744,0.806256,9.704564e-01,-1.470942,-1.465874,0.460299,-1.083173,1,Moderate Risk,1,1
3,Grasberg,0.999596,8.161065e-07,-0.011148,1.011148,9.996033e-01,0.627735,0.775302,0.519157,0.650290,0,High Risk,0,2


In [4]:
weight_candidates = []

step = 0.05

for w1 in np.arange(0.1, 0.81, step):
    for w2 in np.arange(0.1, 0.81, step):
        w3 = 1 - w1 - w2
        if 0.1 <= w3 <= 0.8:
            weight_candidates.append((round(w1,2),
                                      round(w2,2),
                                      round(w3,2)))

len(weight_candidates)

105

In [5]:
results = []

X_base = feature_df[[
    "F1_exposure_intensity_z",
    "F2_vegetation_suppression_z",
    "F3_persistence_raw_z"
]].values

for w1, w2, w3 in weight_candidates:

    composite = (
        w1 * feature_df["F1_exposure_intensity_z"] +
        w2 * feature_df["F2_vegetation_suppression_z"] +
        w3 * feature_df["F3_persistence_raw_z"]
    ).values.reshape(-1, 1)

    # Use composite score + original features
    X = np.column_stack([X_base, composite])

    kmeans = KMeans(n_clusters=3, random_state=42)
    labels = kmeans.fit_predict(X)

    score = silhouette_score(X, labels)

    results.append({
        "w1": w1,
        "w2": w2,
        "w3": w3,
        "silhouette": score
    })

results_df = pd.DataFrame(results)
results_df.sort_values("silhouette", ascending=False).head()

,w1,w2,w3,silhouette
0,0.10,0.10,0.80,0.433983
14,0.15,0.10,0.75,0.432744
1,0.10,0.15,0.75,0.432264
27,0.20,0.10,0.70,0.431498
15,0.15,0.15,0.70,0.431011


In [6]:
best_weights = results_df.sort_values("silhouette", ascending=False).iloc[0]
best_weights

,0
w1,0.100000
w2,0.100000
w3,0.800000
silhouette,0.433983


In [7]:
w1_opt = 0.10
w2_opt = 0.10
w3_opt = 0.80

feature_df["CERI_v3"] = (
    w1_opt * feature_df["F1_exposure_intensity_z"] +
    w2_opt * feature_df["F2_vegetation_suppression_z"] +
    w3_opt * feature_df["F3_persistence_raw_z"]
)

feature_df.sort_values("CERI_v3", ascending=False)[
    ["asset", "CERI_z", "CERI_v3"]
]

,asset,CERI_z,CERI_v3
3,Grasberg,0.650290,0.555630
0,Bingham,0.545452,0.521126
2,Gevra,-1.083173,0.074557
1,Carajás,-0.112568,-1.151313




---


# Ranking Stability Analysis: CERI v2 vs CERI v3

To evaluate the material impact of weight optimization,
we quantify:

- Rank change per asset
- Absolute score change
- Spearman rank correlation
- Kendall Tau correlation

This determines whether optimization alters
portfolio ordering in a structurally meaningful way.

In [8]:
from scipy.stats import spearmanr, kendalltau

# Rank both versions
feature_df["rank_v2"] = feature_df["CERI_z"].rank(ascending=False)
feature_df["rank_v3"] = feature_df["CERI_v3"].rank(ascending=False)

# Rank difference
feature_df["rank_shift"] = (
    feature_df["rank_v3"] - feature_df["rank_v2"]
)

# Absolute score difference
feature_df["score_delta"] = (
    feature_df["CERI_v3"] - feature_df["CERI_z"]
)

feature_df[[
    "asset",
    "rank_v2",
    "rank_v3",
    "rank_shift",
    "score_delta"
]]

,asset,rank_v2,rank_v3,rank_shift,score_delta
0,Bingham,2.0,2.0,0.0,-0.024325
1,Carajás,3.0,4.0,1.0,-1.038746
2,Gevra,4.0,3.0,-1.0,1.157731
3,Grasberg,1.0,1.0,0.0,-0.094660


In [10]:
spearman_corr, _ = spearmanr(
    feature_df["rank_v2"],
    feature_df["rank_v3"]
)

kendall_corr, _ = kendalltau(
    feature_df["rank_v2"],
    feature_df["rank_v3"]
)

print("Spearman Correlation:", spearman_corr)
print("Kendall Tau:", kendall_corr)

Spearman Correlation: 0.7999999999999999
Kendall Tau: 0.6666666666666669


### Phase 6 Summary

Weight optimization improves geometric separation
but does not materially alter extreme-tier ranking.

The exposure scoring framework demonstrates:

- Structural robustness
- Controlled sensitivity
- Stable tier boundaries
- Predictable behavior under weight perturbation

CERI v2 remains governance baseline.
CERI v3 functions as analytical validation.